# Tome 10, Chapter 7: Large deviations (Python companion)

Exercises first, solutions at the end. Rate functions by numerical Legendre transform, **importance sampling** for probabilities of order $10^{-15}$ (the computer science and risk-management application of Cramér's tilt), exact computations with dice and types, the Gibbs conditioning principle seen converging, the error exponent of a statistical test, and the Curie–Weiss magnet. Requires `numpy` and `scipy`. All logarithms are natural.

In [ ]:
import math
import numpy as np
from scipy import stats, optimize, special
rng = np.random.default_rng(70)

## Exercises

**P7.1 (★) Rate functions and the prefactor.** Compute $\Lambda^*(x)=\sup_\theta(\theta x-\Lambda(\theta))$ numerically for the exponential law of mean $1$ and for Poisson$(1)$ at $x=0.5,2,3$, and compare with the closed forms of Exercise 7.1. Then, for $n$ exponential claims and $a=2$, compare the exact tail $P(S_n\ge2n)$ (a gamma tail) with $e^{-n\Lambda^*(2)}$, with the Bahadur–Rao formula, and with the central limit theorem, for $n=10,100,1000$.

**P7.2 (★★) Importance sampling.** Estimate $p=P(S_{100}\ge200)$ for exponential claims of mean $1$ ($p\approx1.84\times10^{-15}$) by Monte Carlo. (a) Why is naive simulation hopeless? (b) Simulate instead under the tilted law $\mu_\theta$ with $\Lambda'(\theta)=2$ (an exponential law of mean $2$), and reweight each sample by $e^{-\theta S_n+n\Lambda(\theta)}$. With $10^4$ samples, what relative error do you get? (c) Try other tilts: which one minimizes the variance, roughly?

**P7.3 (★★) Types, exactly.** For $n$ fair dice, compute exactly $P(\text{average}\ge4.5)$ for $n=10,50,100,500$ (convolve), and compare $\frac1n\log P$ with $-\Lambda^*(4.5)$. Check the two-sided bound of Lemma 7.4(iv) on the single most likely type in $\{\text{average}\ge4.5\}$ for $n=60$.

**P7.4 (★★) Watching the Gibbs conditioning principle.** For $n$ fair dice, compute *exactly* the law of the first die given that the sum is at least $4.5n$, for $n=10,100,1000$: $P(X_1=a\mid S_n\ge4.5n)=\frac{P(X_1=a)P(S_{n-1}\ge4.5n-a)}{P(S_n\ge4.5n)}$. Compare with the Gibbs law $\nu^*$ of Example 7.8, and compute the total variation distance.

**P7.5 (★★) CS application: the price of a decision.** We must tell a fair coin from one with $P(\text{heads})=0.6$ (think of an A/B test, a biased random number generator, or a spam feature). (a) For each $n$ up to $3000$, compute exactly $\beta_n(0.05)$, the smallest probability of accepting the biased coin among tests that reject the fair one with probability at most $0.05$. (By the Neyman–Pearson lemma, the optimal tests declare ``fair'' when the number of heads is below a threshold; allow randomization at the threshold.) Compare $\frac1n\log\beta_n$ with $-D(\mu_0\|\mu_1)$. (b) How many tosses make $\beta_n\le10^{-6}$? (c) Compute the Chernoff information of Exercise 7.6.

**P7.6 (★★★) The magnet.** For the Curie–Weiss model with $n=2000$ spins, compute the exact law of $m_n$ (the number of $+$ spins is binomial, reweighted by $e^{\beta n m^2/2}$) for $\beta=0.8,1,1.2,1.5$. Plot or tabulate it; compare $\mathbb E|m_n|$ with $m^*(\beta)$. At $\beta=1$, compare the law of $n^{1/4}m_n$ with the density proportional to $e^{-u^4/12}$ (Ellis and Newman).

## Solutions

### P7.1
**How to find it.** $\theta\mapsto\theta x-\Lambda(\theta)$ is concave, so a bounded scalar maximization finds the supremum. For the exponential law $\Lambda=+\infty$ for $\theta\ge1$, so search in $(-\infty,1)$. The Bahadur–Rao formula needs $\theta_a=1-\frac1a$ and $\sigma_a^2=\Lambda''(\theta_a)=a^2$ (the tilted law is exponential of mean $a$).

In [ ]:
def legendre(Lam, x, lo, hi):
    r = optimize.minimize_scalar(lambda t: -(t * x - Lam(t)), bounds=(lo, hi), method='bounded', options={'xatol': 1e-12})
    return -r.fun
Lexp = lambda t: -math.log(1 - t)
Lpois = lambda t: math.expm1(t)
for x in (0.5, 2, 3):
    print(f"x = {x}: exponential  numeric {legendre(Lexp, x, -50, 1 - 1e-12):.6f}  formula {x - 1 - math.log(x):.6f}"
          f"   | Poisson(1) numeric {legendre(Lpois, x, -50, 10):.6f}  formula {x * math.log(x) - x + 1:.6f}")

a = 2.0; I = a - 1 - math.log(a); th, sig = 1 - 1 / a, a
print("\n   n     exact          e^{-n I}       Bahadur-Rao    CLT")
for n in (10, 100, 1000):
    exact = stats.gamma.sf(n * a, n)
    print(f"{n:5d}  {exact:.4e}    {math.exp(-n * I):.4e}    {math.exp(-n * I) / (th * sig * math.sqrt(2 * math.pi * n)):.4e}    {stats.norm.sf((a - 1) * math.sqrt(n)):.4e}")

The exponent alone is off by a factor of order $\sqrt n$ (a factor $25$ at $n=100$), the Bahadur–Rao formula is within $2\%$ at $n=100$, and the CLT is wrong by a factor that grows exponentially, $10^8$ at $n=100$.

### P7.2
**How to find it.** (a) The expected number of hits in $N$ samples is $Np\approx N\cdot2\times10^{-15}$. (b) Under $\mu_\theta^{\otimes n}$ the event is typical, and the identity $p=\mathbb E_\theta[e^{-\theta S_n+n\Lambda(\theta)}\mathbf 1\{S_n\ge na\}]$ is exact, so the reweighted average is unbiased. On the event, the weight is at most $e^{-n\Lambda^*(a)}$, so the relative variance stays bounded by a power of $n$ instead of growing like $1/p$.

In [ ]:
n, a, N = 100, 2.0, 10**4
exact = stats.gamma.sf(n * a, n)
def tilted_estimate(mean, N):
    theta = 1 - 1 / mean                      # exponential of mean `mean` is the tilt with this theta
    S = rng.exponential(mean, (N, n)).sum(axis=1)
    w = np.exp(-theta * S - n * math.log(1 - theta)) * (S >= n * a)
    return w.mean(), w.std() / math.sqrt(N)
print("exact p =", exact)
print("naive: number of hits in 10^6 samples:", int((rng.exponential(1, (10**6 // 100, n)).sum(axis=1) >= n * a).sum()), "(of 10^4 sums)")
for mean in (1.5, 1.8, 2.0, 2.2, 2.5):
    est, se = tilted_estimate(mean, N)
    print(f"tilt to mean {mean}: estimate {est:.4e}, relative std error {se / est:.3f}, true relative error {abs(est - exact) / exact:.3f}")

With $10^4$ tilted samples the relative standard error is $3$–$5\%$ for tilts between means $1.8$ and $2.2$, around the Cramér tilt (mean $2$), and grows quickly when the tilt is too weak (mean $1.5$: about $45\%$, the event stays rare under the new law) or too strong (mean $2.5$: about $10\%$). This is how rare-event probabilities are computed in reliability, finance and network engineering, and how Esscher's actuarial trick lives on in software.

### P7.3
**How to find it.** Convolve the uniform law on $\{1,\dots,6\}$ with itself $n$ times (integer arithmetic is exact with Python integers, but floats are fine here). For the single type, maximize $D(\nu\|\mu)$'s negative over types with average $\ge4.5$ by brute force for $n=60$.

In [ ]:
k = np.arange(1, 7)
theta = optimize.brentq(lambda t: (k * np.exp(t * k)).sum() / np.exp(t * k).sum() - 4.5, -5, 5)
Lstar = theta * 4.5 - math.log(np.exp(theta * k).mean())
print("theta =", theta, " Lambda*(4.5) =", Lstar)
die = np.r_[0, np.ones(6) / 6]
def sum_law(n):
    d = np.array([1.0])
    for _ in range(n):
        d = np.convolve(d, die)
    return d                                   # d[s] = P(S_n = s)
for n in (10, 50, 100, 500):
    d = sum_law(n); P = d[math.ceil(4.5 * n):].sum()
    print(f"n = {n:4d}: P = {P:.3e},  (1/n) log P = {math.log(P) / n:.4f},  -Lambda* = {-Lstar:.4f}")

# most likely type with average >= 4.5, n = 60, by brute force over compositions
import itertools
n = 60; best = None
for c in itertools.product(range(n + 1), repeat=5):
    s = sum(c)
    if s > n: continue
    cnt = np.array(c + (n - s,))
    if (cnt * k).sum() < 4.5 * n: continue
    nu = cnt / n
    D = sum(p * math.log(6 * p) for p in nu if p > 0)
    if best is None or D < best[0]:
        best = (D, cnt)
D, cnt = best
logP = special.gammaln(n + 1) - special.gammaln(cnt + 1).sum() - n * math.log(6)
print("\nmost likely type (counts of 1..6):", cnt, " D =", round(D, 4))
print("log P(L_n = nu) =", round(logP, 3), " bounds:", round(-n * D - 6 * math.log(n + 1), 3), "<=  ...  <=", round(-n * D, 3))

The rate converges slowly, like $\frac{\log n}n$: the polynomial prefactor. The most likely type at $n=60$ has counts close to $60\,\nu^*$, and its probability sits between the two bounds of Lemma 7.4, closer to the upper one: the factor $(n+1)^{-|A|}$ is a crude worst case. (The brute force loop takes a few seconds.)

### P7.4
**How to find it.** Condition on $X_1$: everything is a convolution, computed once per $n$.

In [ ]:
k = np.arange(1, 7)
nu_star = np.exp(theta * k) / np.exp(theta * k).sum()
print("nu* =", nu_star.round(4))
for n in (10, 100, 1000):
    d_prev = sum_law(n - 1)
    tail = lambda s: d_prev[max(0, s):].sum()
    thr = math.ceil(4.5 * n)
    cond = np.array([tail(thr - a) for a in k]) / 6
    cond /= cond.sum()
    print(f"n = {n:4d}: law of X_1 given the event = {cond.round(4)},  TV to nu* = {0.5 * np.abs(cond - nu_star).sum():.4f}")

The conditional law moves towards the geometric tilt, with a TV distance shrinking like $1/n$ (the effect of the integer threshold and of finite $n$). Nothing in the computation mentions entropy, and yet the answer is the maximum-entropy law of Jaynes.

### P7.5
**How to find it.** With $K$ heads in $n$ tosses, the likelihood ratio is monotone in $K$, so the optimal test with first error exactly $0.05$ declares "fair" when $K<t$, and with probability $\gamma$ when $K=t$, where $t,\gamma$ solve $P_0(K>t)+(1-\gamma)P_0(K=t)=0.05$ — the randomization makes the first error exactly $0.05$. Then $\beta_n=P_1(K<t)+\gamma P_1(K=t)$.

In [ ]:
p0, p1, eps = 0.5, 0.6, 0.05
D01 = p0 * math.log(p0 / p1) + (1 - p0) * math.log((1 - p0) / (1 - p1))
def beta(n):
    K = np.arange(n + 1)
    sf0 = stats.binom.sf(K, n, p0)            # P0(K > t)
    t = int(np.argmax(sf0 <= eps))             # smallest t with P0(K > t) <= eps
    pt0 = stats.binom.pmf(t, n, p0)
    gamma = 1 - (eps - sf0[t]) / pt0           # declare 'fair' with prob gamma at K = t
    return stats.binom.cdf(t - 1, n, p1) + gamma * stats.binom.pmf(t, n, p1)
print("D(mu0||mu1) =", D01)
for n in (100, 300, 1000, 3000):
    b = beta(n)
    print(f"n = {n:5d}: beta_n = {b:.3e},  (1/n) log beta_n = {math.log(b) / n:.5f}")
n = 100
while beta(n) > 1e-6:
    n += 10
print("smallest n (step 10) with beta_n <= 1e-6:", n, "  leading-order estimate log(1e6)/D =", round(math.log(1e6) / D01))
r = optimize.minimize_scalar(lambda s: math.log(p0**(1 - s) * p1**s + (1 - p0)**(1 - s) * (1 - p1)**s), bounds=(0, 1), method='bounded')
print("Chernoff information C =", -r.fun, "at s =", r.x)

$\frac1n\log\beta_n$ approaches $-0.0204$ slowly from above: the correction is of order $n^{-1/2}$, because the first error $0.05$ forces the threshold $1.645\sqrt n/2$ heads above $n/2$. So the leading-order estimate of $680$ tosses is optimistic: about $1000$ are needed. Exponents tell you how costs scale, not what they are at moderate $n$; for that, compute.

### P7.6
**How to find it.** $m_n=\frac{2K-n}n$ with $K$ binomial$(n,\frac12)$ under fair signs; the Curie–Weiss law multiplies $P(K=j)$ by $e^{\beta nm^2/2}$ and renormalizes. Work with logarithms to avoid overflow.

In [ ]:
n = 2000
K = np.arange(n + 1); m = (2 * K - n) / n
logb = stats.binom.logpmf(K, n, 0.5)
def cw_law(beta):
    lw = logb + beta * n * m**2 / 2
    w = np.exp(lw - lw.max())
    return w / w.sum()
for beta in (0.8, 1.0, 1.2, 1.5):
    w = cw_law(beta)
    mstar = optimize.brentq(lambda y: y - math.tanh(beta * y), 1e-9, 1) if beta > 1 else 0.0
    print(f"beta = {beta}: E|m_n| = {(w * np.abs(m)).sum():.4f},  m* = {mstar:.4f},  P(m_n > 0) = {w[m > 0].sum():.3f}")

w = cw_law(1.0)
u = n**0.25 * m
dens = np.exp(-u**4 / 12)
dens /= np.trapezoid(dens, u)
# compare the probability of |n^{1/4} m_n| <= t with the limit
for t in (0.5, 1.0, 2.0, 3.0):
    lim = np.trapezoid(dens[np.abs(u) <= t], u[np.abs(u) <= t])
    print(f"beta = 1: P(|n^(1/4) m_n| <= {t}) = {w[np.abs(u) <= t].sum():.4f},  limit {lim:.4f}")

At $\beta=0.8$, $\mathbb E|m_n|$ is of order $n^{-1/2}$: no magnetization. At $\beta=1.2$ and $1.5$ it is close to $m^*(\beta)$, while $P(m_n>0)=\frac12$: the system is magnetized, in a random direction. At $\beta=1$ the magnetization lives at the scale $n^{-1/4}$, and the law of $n^{1/4}m_n$ is already close to the quartic-exponential density at $n=2000$.